# 06 · An agent drives the SO-101 through its tool

**What you will see**
- `robot.tool` and `sim.tool`: strands tools generated from the robot's manifest and the world's public verbs, not written by hand.
- A scripted (offline) model calls `observe`, then `act`, and the arm's `shoulder_pan` moves.
- One real Claude turn on Bedrock doing the same thing from a plain-English request, with a video of the move.
- The operator gate: a hardware SO-101 refuses an agent's `act` unless the operator's `confirm=` callable says yes.

The G1 humanoid stands next to the arm in the same world the whole time.

In [1]:
import _nb
_nb.drawing('v07_dashboard')

## One world, two robots, three tools

In [2]:
import json
from strands_robots import Robot, Simulation

sim = Simulation()
arm = Robot("so101", sim=sim)
g1 = Robot("unitree_g1", sim=sim, position=[1.0, 0.0, 0.0])
for t in (arm.tool, g1.tool, sim.tool):
    print(f"{t.tool_name:18s} rpcs: {t.tool_spec['inputSchema']['json']['properties']['rpc']['enum']}")

robot_so101        rpcs: ['describe', 'observe', 'act', 'run', 'stop', 'hold', 'estop', 'calibrate']
robot_unitree_g1   rpcs: ['describe', 'observe', 'act', 'run', 'stop', 'hold', 'estop', 'calibrate']
sim_world          rpcs: ['add_robot', 'add_object', 'camera', 'contacts', 'control_step', 'describe', 'randomize', 'remove', 'render', 'reset', 'set_pose', 'step']


Each robot tool has one `rpc` selector plus one argument object per rpc. `resume` is missing on purpose: only the operator can release an e-stop. Here is the schema the model sees for `act`, built from `Robot.act`'s signature and the arm's joint names:

In [3]:
spec = arm.tool.tool_spec
print(spec["description"])
print(json.dumps(spec["inputSchema"]["json"]["properties"]["act"], indent=2))

so101: call one rpc; its arguments go under the property named after it. Joints: ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper'].
{
  "type": "object",
  "properties": {
    "action": {
      "type": "object",
      "properties": {
        "shoulder_pan": {
          "type": "number"
        },
        "shoulder_lift": {
          "type": "number"
        },
        "elbow_flex": {
          "type": "number"
        },
        "wrist_flex": {
          "type": "number"
        },
        "wrist_roll": {
          "type": "number"
        },
        "gripper": {
          "type": "number"
        }
      },
      "additionalProperties": false,
      "description": "joint -> target, any subset of ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']"
    },
    "steps": {
      "type": [
        "integer",
        "null"
      ]
    }
  },
  "required": [
    "action"
  ]
}


The world tool takes placement arguments only. A model can add a robot to the scene, but it can never mount hardware (no `port`, `driver` or `confirm`):

In [4]:
print(json.dumps(sim.tool.tool_spec["inputSchema"]["json"]["properties"]["add_robot"], indent=2))

{
  "type": "object",
  "properties": {
    "name": {
      "type": "string"
    },
    "position": {
      "type": [
        "array",
        "null"
      ],
      "items": {
        "type": "number"
      },
      "minItems": 3,
      "maxItems": 3
    },
    "orientation": {
      "type": [
        "array",
        "null"
      ],
      "items": {
        "type": "number"
      },
      "minItems": 4,
      "maxItems": 4
    },
    "keyframe": {
      "type": [
        "string",
        "null"
      ]
    },
    "id": {
      "type": [
        "string",
        "null"
      ]
    }
  },
  "required": [
    "name"
  ]
}


## A scripted turn: observe, then act (no network)

A tiny strands `Model` that plays back two tool calls. It's the same kind of model the test suite uses.

In [5]:
import json as _json
from strands import Agent
from strands.models.model import Model


class ScriptedModel(Model):
    """Plays back a fixed list of tool calls, then says 'done'."""

    def __init__(self, calls):
        self.calls = list(calls)

    def update_config(self, **kw):
        pass

    def get_config(self):
        return {}

    async def structured_output(self, *a, **k):
        raise NotImplementedError
        yield

    async def stream(self, messages, tool_specs=None, system_prompt=None, **kw):
        yield {"messageStart": {"role": "assistant"}}
        if self.calls:
            name, inp = self.calls.pop(0)
            yield {"contentBlockStart": {"start": {"toolUse": {"toolUseId": f"t{len(self.calls)}", "name": name}}}}
            yield {"contentBlockDelta": {"delta": {"toolUse": {"input": _json.dumps(inp)}}}}
            yield {"contentBlockStop": {}}
            yield {"messageStop": {"stopReason": "tool_use"}}
        else:
            yield {"contentBlockDelta": {"delta": {"text": "done"}}}
            yield {"contentBlockStop": {}}
            yield {"messageStop": {"stopReason": "end_turn"}}


def tool_trace(agent):
    uses = [b["toolUse"] for m in agent.messages for b in m["content"] if "toolUse" in b]
    results = [b["toolResult"] for m in agent.messages for b in m["content"] if "toolResult" in b]
    return [(u["name"], u["input"], r["status"]) for u, r in zip(uses, results)]


before = arm.observe(images=False)["shoulder_pan"]
model = ScriptedModel([
    ("robot_so101", {"rpc": "observe"}),
    ("robot_so101", {"rpc": "act", "act": {"action": {"shoulder_pan": 0.4}, "steps": 200}}),
])
agent = Agent(model=model, tools=[arm.tool, g1.tool, sim.tool], callback_handler=None)
print("reply:", agent("move the base joint"))
for name, inp, status in tool_trace(agent):
    print(f"  {name} {inp} -> {status}")
print(f"shoulder_pan {before:+.3f} -> {arm.observe(images=False)['shoulder_pan']:+.3f} rad (target 0.4)")

reply: done

  robot_so101 {'rpc': 'observe'} -> success
  robot_so101 {'rpc': 'act', 'act': {'action': {'shoulder_pan': 0.4}, 'steps': 200}} -> success
shoulder_pan +0.000 -> +0.375 rad (target 0.4)


Errors go back to the model as text rather than as exceptions, so it can correct itself:

In [6]:
agent = Agent(model=ScriptedModel([
    ("robot_so101", {"rpc": "act", "act": {"action": {"elbow": 1.0}}}),
    ("robot_so101", {"rpc": "fly"}),
]), tools=[arm.tool], callback_handler=None)
agent("try things")
for m in agent.messages:
    for b in m["content"]:
        if "toolResult" in b:
            print(b["toolResult"]["status"], "|", b["toolResult"]["content"][0]["text"][:160])

error | KeyError: "unknown joint(s) ['elbow'] for so101; action_keys are ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']"
error | ValueError: unknown rpc 'fly'; so101 offers ['describe', 'observe', 'act', 'run', 'stop', 'hold', 'estop', 'calibrate']


## One real model turn (Claude Sonnet 4.5 on Bedrock, us-west-2)

One fresh world (SO-101 + G1), one request in plain English. If Bedrock is unreachable, the cell prints why and carries on.

In [7]:
import time
MODEL_ID, REGION = "us.anthropic.claude-sonnet-4-5-20250929-v1:0", "us-west-2"

live = Simulation()
live_arm = Robot("so101", sim=live)
Robot("unitree_g1", sim=live, position=[1.0, 0.0, 0.0])
live_calls, live_error = [], None
try:
    from strands.models.bedrock import BedrockModel

    bedrock = BedrockModel(model_id=MODEL_ID, region_name=REGION, max_tokens=512, temperature=0.0)
    real_agent = Agent(model=bedrock, tools=[live_arm.tool], callback_handler=None,
                       system_prompt="You control a simulated robot arm through one tool. Be brief.")
    t0 = time.perf_counter()
    reply = real_agent("First observe the robot. Then act once to set shoulder_pan to 0.5 radians with steps=200. "
                       "Reply with the final shoulder_pan value.")
    dt = time.perf_counter() - t0
    live_calls = tool_trace(real_agent)
    for name, inp, status in live_calls:
        print(f"tool call: {name} {inp} -> {status}")
    print(f"model reply ({dt:.1f} s): {str(reply).strip()}")
    u = real_agent.event_loop_metrics.accumulated_usage
    print("tokens:", u.get("totalTokens"))
except Exception as exc:  # noqa: BLE001
    live_error = f"{type(exc).__name__}: {exc}"
    print("Bedrock turn did not run:", live_error)
print(f"final shoulder_pan: {live_arm.observe(images=False)['shoulder_pan']:+.4f} rad")

tool call: robot_so101 {'rpc': 'observe', 'observe': {}} -> success
tool call: robot_so101 {'act': {'action': {'shoulder_pan': 0.5}, 'steps': 200}, 'rpc': 'act'} -> success
model reply (6.9 s): The final shoulder_pan value is **0.469 radians** (approximately 0.47 radians).
tokens: 5554
final shoulder_pan: +0.4692 rad


The video replays the model's own `act` call in the same world: we reset it, then apply the call's action in 10-step slices and render a frame after each slice. On macOS, OpenGL renders only on the thread that owns the context, so we render after the turn, not during it.

In [8]:
act_calls = [inp["act"] for name, inp, status in live_calls if inp.get("rpc") == "act" and status == "success"]
act = act_calls[-1] if act_calls else {"action": {"shoulder_pan": 0.5}, "steps": 200}  # the request, if the turn failed
live.reset()
live.camera("front", position=(0.9, -1.4, 0.9), target=(0.5, 0.0, 0.35), size=(640, 400))
frames = [live.render("front")]
for _ in range((act.get("steps") or 200) // 10):
    live_arm.act(act["action"], steps=10)
    frames.append(live.render("front"))
print("replayed", act, f"-> shoulder_pan {live_arm.observe(images=False)['shoulder_pan']:+.4f}")
_nb.video(frames, fps=15)

replayed {'action': {'shoulder_pan': 0.5}, 'steps': 200} -> shoulder_pan +0.4686


## The operator gate: hardware never moves because a model said so

A hardware SO-101 on a simulated Feetech bus (six fake STS3215 servos from the test suite; no USB, no motion). Plain Python `act` is the developer's own call. An **agent's** `act` goes through the safety chain and then the operator gate: without `confirm=`, it's refused and no goal frame reaches the bus.

In [9]:
import sys
sys.path.insert(0, str(_nb.ROOT / "tests"))
from test_drivers import CAL, FakeSerial  # a byte-level fake of the servo bus
from strands_robots import Description
from strands_robots.robot.drivers import feetech as ft


def hardware_arm(confirm=None):
    serial = FakeSerial()
    driver = ft.FeetechDriver(Description.load("so101"), port="/dev/fake", calibration=CAL,
                              bus=ft.FeetechBus("/dev/fake", serial=serial))
    return Robot("so101", driver=driver, twin=False, confirm=confirm), serial


def goal_frames(serial):
    return sum(1 for f in serial.frames if f[4] == ft.SYNC_WRITE and f[5] in (ft.REG_GOAL_POSITION, ft.REG_TORQUE_ENABLE))


def agent_act(robot, **extra):
    a = Agent(model=ScriptedModel([("robot_so101", {"rpc": "act", "act": {"action": {"shoulder_pan": 0.2}, **extra}})]),
              tools=[robot.tool], callback_handler=None)
    a("go")
    r = [b["toolResult"] for m in a.messages for b in m["content"] if "toolResult" in b][0]
    return r["status"], r["content"][0]["text"]


real, serial = hardware_arm()
print("is_real:", real.is_real)
n = goal_frames(serial); status, text = agent_act(real)
print(f"no operator      -> {status}: {text[:140]}\n   goal frames written: {goal_frames(serial) - n}")
n = goal_frames(serial); status, text = agent_act(real, confirmed=True)
print(f"model says 'yes' -> {status}: {text[:140]}\n   goal frames written: {goal_frames(serial) - n}")

asked = []
real2, serial2 = hardware_arm(confirm=lambda targets: asked.append(targets) or True)
n = goal_frames(serial2); status, text = agent_act(real2)
print(f"operator says yes -> {status}; operator asked {len(asked)}x; goal frames written: {goal_frames(serial2) - n}")
print("   shoulder_pan now", round(real2.observe(images=False)["shoulder_pan"], 3))
real2, serial2 = hardware_arm(confirm=lambda targets: False)
n = goal_frames(serial2); status, text = agent_act(real2)
print(f"operator says no  -> {status}: {text[:100]}; goal frames written: {goal_frames(serial2) - n}")

is_real: True
no operator      -> error: RequestRefused: so101 is a real robot and this command came through an agent or dashboard: pass confirm=<callable(action) -> bool> to Robot(
   goal frames written: 0
model says 'yes' -> error: TypeError: act does not take ['confirmed']; it takes ['action', 'steps']
   goal frames written: 0
operator says yes -> success; operator asked 1x; goal frames written: 2
   shoulder_pan now 0.201
operator says no  -> error: RequestRefused: so101: the operator gate refused this action; goal frames written: 0


In [10]:
for r in (real, real2):
    r.close()
for s in (sim, live):
    s.close()

## Where this sits in v1
- **Part 8 (tools):** `robot.tool` / `sim.tool` are generated from the device manifest and the world's public verbs, so adding an rpc changes the tool with no tool code. strands is imported only when you touch `.tool`.
- **Part 5.5 (device):** every agent write goes through the same safety chain as the dashboard, then the operator gate on hardware. An agent can e-stop but never resume.
- Next: the same tools behind the dashboard's chat panel (07) and on mesh devices.